# Linear Regression  Life Insurance Premium Prediction
### Predicting Annual Premium Amount using Applicant Profile Data

---

##  Problem Statement

An insurance company wants to **automate the calculation of annual life insurance premiums** for new applicants based on their personal, financial, and lifestyle profile  instead of relying entirely on manual underwriting.

**Business Goal:** Build a regression model that predicts the **annual premium amount (₹)** a customer should be charged, based on factors like age, BMI, smoking habit, income, existing medical conditions, and policy details.

**Why this matters:**
- Insurers price premiums using **actuarial risk factors**  older age, smoking, high BMI, and pre-existing conditions increase risk and therefore premium.
- A data-driven pricing model makes premium calculation **faster, consistent, and explainable**, instead of varying case-by-case based on manual judgment.
- Understanding which factors drive the premium up or down helps the company **design better policies** and helps customers understand **why they're charged what they're charged**.

---


## Step 1 Data understanding


## Dataset Variables (Real-World Insurance Columns)

| Column | Type | Real-World Meaning | Risk Direction |
|---|---|---|---|
| `age` | Numerical | Applicant's age in years | Older → higher premium |
| `bmi` | Numerical | Body Mass Index (weight/height²) | Higher BMI → higher health risk → higher premium |
| `annual_income` | Numerical | Applicant's yearly income (₹) | Higher income → can buy bigger sum assured → higher premium |
| `sum_assured` | Numerical | The payout amount the policy guarantees (₹) | Higher sum assured → higher premium |
| `policy_term_years` | Numerical | Duration of the policy (years) | Longer term → different premium structure |
| `num_dependents` | Numerical | Number of dependents (spouse/children) | More dependents → typically higher coverage needs |
| `smoker` | Categorical | Yes / No — whether applicant smokes | Smokers pay significantly more (major actuarial factor) |
| `medical_history` | Categorical | None / Minor / Major pre-existing condition | Worse history → higher premium |
| `annual_premium` | **Target** | The amount (₹) the insurer charges per year | What we want to predict |



## Step 2  Import Libraries

In [1]:
print('hii')

hii


In [2]:
!pip install numpy pandas scikit-learn matplotlib seaborn

   ---------------------------------------- 0.0/12.6 MB ? eta -:--:--
   ---- ----------------------------------- 1.3/12.6 MB 5.8 MB/s eta 0:00:02
   -------- ------------------------------- 2.6/12.6 MB 6.3 MB/s eta 0:00:02
   ----------- ---------------------------- 3.7/12.6 MB 5.8 MB/s eta 0:00:02
   ---------------- ----------------------- 5.2/12.6 MB 6.1 MB/s eta 0:00:02
   --------------------- ------------------ 6.8/12.6 MB 6.4 MB/s eta 0:00:01
   ------------------------- -------------- 8.1/12.6 MB 6.4 MB/s eta 0:00:01
   ------------------------------ --------- 9.7/12.6 MB 6.6 MB/s eta 0:00:01
   ----------------------------------- ---- 11.0/12.6 MB 6.5 MB/s eta 0:00:01
   ---------------------------------------  12.3/12.6 MB 6.6 MB/s eta 0:00:01
   ---------------------------------------- 12.6/12.6 MB 6.4 MB/s  0:00:01
   ---------------------------------------- 0.0/9.9 MB ? eta -:--:--
   ------- -------------------------------- 1.8/9.9 MB 8.5 MB/s eta 0:00:01
   ------------

In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import mean_squared_error, r2_score
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.facecolor': '#FAFAFA',
    'axes.facecolor':   '#F4F6F9',
    'axes.grid':        True,
    'grid.alpha':       0.4,
    'font.size':        12,
})
sns.set_palette("husl")
print("All libraries imported!")


Matplotlib is building the font cache; this may take a moment.


All libraries imported!


## Step 3 — Exploratory Data Analysis (EDA)

### Why EDA?
Before cleaning or modelling, we need to **understand the raw data**:
- What's the structure and data types?
- Where and how much data is missing?
- Are there extreme/unrealistic values (outliers)?
- How does each feature relate to the premium?

>  **Rule:** Never clean what you haven't explored. EDA tells you *what* to clean and *why*.


In [4]:
df=pd.read_csv('life_insurance.csv')

In [5]:
print("=" * 55)
print("DATASET INFO")
print("=" * 55)
df.info()


DATASET INFO
<class 'pandas.DataFrame'>
RangeIndex: 600 entries, 0 to 599
Data columns (total 11 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   age                600 non-null    int64  
 1   bmi                565 non-null    float64
 2   annual_income      572 non-null    float64
 3   sum_assured        600 non-null    int64  
 4   policy_term_years  600 non-null    int64  
 5   num_dependents     600 non-null    int64  
 6   smoker             600 non-null    str    
 7   medical_history    194 non-null    str    
 8   applicant_id       600 non-null    int64  
 9   lucky_number       600 non-null    int64  
 10  annual_premium     600 non-null    float64
dtypes: float64(3), int64(6), str(2)
memory usage: 51.7 KB


In [ ]:
print("=" * 55)
print("STATISTICAL SUMMARY — Numerical Features")
print("=" * 55)
df.describe().round(2)


In [ ]:
print("=" * 55)
print("MISSING VALUES COUNT")
print("=" * 55)
missing     = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df  = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
missing_df  = missing_df[missing_df['Missing Count'] > 0]
print(missing_df)

fig, ax = plt.subplots(figsize=(8, 4))
missing_df['Missing %'].plot(kind='bar', ax=ax,
       color=['#E74C3C', '#E67E22', '#9B59B6'], edgecolor='white')
ax.set_title('Missing Values per Feature (%)', fontsize=14, fontweight='bold')
ax.set_ylabel('% Missing'); ax.set_xlabel('')
for i, v in enumerate(missing_df['Missing %']):
    ax.text(i, v + 0.3, f'{v}%', ha='center', fontweight='bold')
plt.xticks(rotation=15); plt.tight_layout()
plt.savefig('missing_values.png', dpi=120); plt.show()


**WHY this matters:** An incomplete BMI or medical history record could bias premium estimates if ignored or dropped carelessly.

In [ ]:
# ── Distribution histograms ─────────────────────────────────
num_cols = ['age', 'bmi', 'annual_income', 'sum_assured',
            'policy_term_years', 'num_dependents', 'lucky_number', 'annual_premium']

fig, axes = plt.subplots(3, 3, figsize=(16, 12))
axes = axes.flatten()
fig.suptitle('Feature Distributions (Before Cleaning)', fontsize=15, fontweight='bold', y=1.01)
palette = sns.color_palette("husl", len(num_cols))

for i, col in enumerate(num_cols):
    axes[i].hist(df[col].dropna(), bins=35, color=palette[i],
                 edgecolor='white', linewidth=0.5, alpha=0.85)
    axes[i].set_title(col, fontweight='bold')
    axes[i].set_ylabel('Frequency')

axes[-1].set_visible(False)
plt.tight_layout()
plt.savefig('feature_distributions.png', dpi=120); plt.show()


Notice annual_income has a long RIGHT TAIL — those are our outliers!

In [ ]:
# ── Box plots — outlier detection ─────────────────────────────
fig, axes = plt.subplots(2, 4, figsize=(18, 9))
axes = axes.flatten()
fig.suptitle(' Box Plots — Outlier Detection (Before Cleaning)', fontsize=15, fontweight='bold')
colors = sns.color_palette("Set2", len(num_cols))

for i, col in enumerate(num_cols):
    axes[i].boxplot(df[col].dropna(), patch_artist=True,
                    boxprops=dict(facecolor=colors[i], alpha=0.7),
                    medianprops=dict(color='black', linewidth=2),
                    flierprops=dict(marker='o', color='red', markersize=6, alpha=0.6))
    axes[i].set_title(col, fontweight='bold')

plt.tight_layout()
plt.savefig('boxplots_before.png', dpi=120); plt.show()
print("\n RED DOTS = outliers. annual_income clearly has extreme values!")


In [ ]:
# ── Categorical feature distributions ─────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('Categorical Feature Distributions', fontsize=14, fontweight='bold')

for ax, col, pal in zip(axes, ['smoker', 'medical_history'], ['Set2', 'Pastel1']):
    counts = df[col].value_counts()
    bars = ax.bar(counts.index, counts.values,
                  color=sns.color_palette(pal, len(counts)), edgecolor='white', linewidth=0.8)
    ax.set_title(col, fontweight='bold'); ax.set_ylabel('Count')
    for bar, val in zip(bars, counts.values):
        ax.text(bar.get_x() + bar.get_width()/2, val + 2, str(val), ha='center', fontweight='bold')

plt.tight_layout(); plt.savefig('categorical_dist.png', dpi=120); plt.show()


In [ ]:
# ── Premium by Smoker Status (business insight) ──────────────
fig, ax = plt.subplots(figsize=(9, 5))
data_to_plot = [
    df[df['smoker'] == 'No']['annual_premium'].values,
    df[df['smoker'] == 'Yes']['annual_premium'].values
]
bp = ax.boxplot(data_to_plot, patch_artist=True, labels=['Non-Smoker', 'Smoker'],
                medianprops=dict(color='black', linewidth=2.5))
bp['boxes'][0].set_facecolor('#2ECC71'); bp['boxes'][0].set_alpha(0.6)
bp['boxes'][1].set_facecolor('#E74C3C'); bp['boxes'][1].set_alpha(0.6)
ax.set_title(' Annual Premium: Smokers vs Non-Smokers', fontsize=13, fontweight='bold')
ax.set_ylabel('Annual Premium (₹)')
plt.tight_layout(); plt.savefig('premium_by_smoker.png', dpi=120); plt.show()

print(f" Avg premium — Non-Smokers : ₹{df[df['smoker']=='No']['annual_premium'].mean():,.0f}")
print(f" Avg premium — Smokers     : ₹{df[df['smoker']=='Yes']['annual_premium'].mean():,.0f}")
print(" This matches real-world underwriting — smoking is one of the strongest risk loadings.")


## Step 4 — Correlation Heatmap (Before Cleaning)

### Why correlation analysis?
- Identifies which features are **strongly linked to premium** (keep them!)
- Identifies features with **near-zero correlation** (candidates for removal)
- Flags **multicollinearity** between features

> **Rule:** A feature with correlation near 0 with the target adds noise, not signal.


In [ ]:
num_df = df[['age','bmi','annual_income','sum_assured','policy_term_years',
             'num_dependents','applicant_id','lucky_number','annual_premium']].dropna()

corr = num_df.corr()

fig, ax = plt.subplots(figsize=(11, 9))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, vmin=-1, vmax=1, ax=ax,
            linewidths=0.5, linecolor='white',
            annot_kws={"size": 10, "weight": "bold"},
            cbar_kws={"shrink": 0.8})
ax.set_title(' Correlation Heatmap (Before Cleaning)', fontsize=14, fontweight='bold', pad=15)
plt.xticks(rotation=45, ha='right'); plt.yticks(rotation=0)
plt.tight_layout(); plt.savefig('correlation_heatmap_before.png', dpi=120); plt.show()

print("\n Correlation with annual_premium:")
print(corr['annual_premium'].drop('annual_premium').sort_values(ascending=False).round(3))
print("\n'applicant_id' and 'lucky_number' have correlation ≈ 0 → IRRELEVANT → DROP!")


## Step 5 — Data Cleaning

### Plan:
| Problem | Feature | Strategy | Why |
|---|---|---|---|
| Outliers | `annual_income` | IQR method | Extreme incomes distort the regression line |
| Missing (numerical) | `bmi`, `annual_income` | Fill with **median** | Robust to skew/outliers |
| Missing (categorical) | `medical_history` | Fill with **mode** | Most representative category |
| Irrelevant features | `applicant_id`, `lucky_number` | **Drop** | Zero correlation → pure noise |


In [ ]:
df_clean = df.copy()

# ── 5A: Outlier Removal ──────────────────────────────────────
print("=" * 55)
print(" 5A: OUTLIER REMOVAL — IQR Method on annual_income")
print("=" * 55)

Q1  = df_clean['annual_income'].quantile(0.25)
Q3  = df_clean['annual_income'].quantile(0.75)
IQR = Q3 - Q1
lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

print(f"  Q1={Q1:.0f}  Q3={Q3:.0f}  IQR={IQR:.0f}")
print(f"  Acceptable range: [{lower:.0f}, {upper:.0f}]")

outliers_mask = (df_clean['annual_income'] < lower) | (df_clean['annual_income'] > upper)
print(f"  Outlier rows detected: {outliers_mask.sum()}")

df_clean = df_clean[~outliers_mask].copy()
print(f"  Rows after outlier removal: {len(df_clean)}")
print("\n WHY IQR? A ₹70,00,000 income entry is statistically extreme for this population")
print(" and would distort the line fit for typical policyholders.")


In [ ]:
# ── 5B: Fill Missing Values ──────────────────────────────────
print("=" * 55)
print(" 5B: FILL MISSING VALUES")
print("=" * 55)

for col in ['bmi', 'annual_income']:
    median_val    = df_clean[col].median()
    missing_count = df_clean[col].isnull().sum()
    df_clean[col] = df_clean[col].fillna(median_val)
    print(f"  '{col}': filled {missing_count} NaN with median = {median_val:.2f}")

mode_val      = df_clean['medical_history'].mode()[0]
missing_count = df_clean['medical_history'].isnull().sum()
df_clean['medical_history'] = df_clean['medical_history'].fillna(mode_val)
print(f"  'medical_history': filled {missing_count} NaN with mode = '{mode_val}'")

print(f"\n Total NaN remaining: {df_clean.isnull().sum().sum()}")
print("\n WHY MEDIAN for BMI/income? Mean is sensitive to skew; median is stable.")
print(" WHY MODE for medical_history? Most applicants have 'None' — the most likely guess.")


In [ ]:
# ── 5C: Drop Irrelevant Features ─────────────────────────────
print("=" * 55)
print(" 5C: DROP IRRELEVANT FEATURES")
print("=" * 55)

drop_cols = ['applicant_id', 'lucky_number']
print(f"  Dropping: {drop_cols}")
print("  Reason: correlation with annual_premium ≈ 0 (confirmed in heatmap)")
print("  An applicant ID is just a label, and lucky_number is random noise —")
print("  no real insurer would use these as pricing factors.")

df_clean.drop(columns=drop_cols, inplace=True)
print(f"\n Final shape after cleaning: {df_clean.shape}")
print(" Remaining features:", df_clean.columns.tolist())


In [ ]:
# ── Box plot — before vs after outlier removal ────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('annual_income — Before vs After Outlier Removal', fontsize=14, fontweight='bold')

for ax, data, title, color in zip(
    axes,
    [df['annual_income'].dropna(), df_clean['annual_income']],
    ['BEFORE (with outliers)', 'AFTER (cleaned)'],
    ['#E74C3C', '#2ECC71']
):
    ax.boxplot(data, patch_artist=True,
               boxprops=dict(facecolor=color, alpha=0.6),
               medianprops=dict(color='black', linewidth=2),
               flierprops=dict(marker='o', color='red', markersize=6, alpha=0.7))
    ax.set_title(title, fontweight='bold')
    ax.set_ylabel('annual_income (₹)')

plt.tight_layout(); plt.savefig('boxplot_before_after.png', dpi=120); plt.show()
print(" Outliers removed. Distribution is now realistic for the applicant pool.")


## Step 6 — Feature Engineering

### Steps:
1. **Label Encoding** — Convert `smoker` and `medical_history` text to numbers
2. **Train-Test Split** — 80% train / 20% test, done *before* scaling
3. **Feature Scaling** — Standardise numerical features (mean=0, std=1)

> **Why scale?** `sum_assured` (in lakhs) and `num_dependents` (0–5) are on wildly different scales — scaling puts every feature on equal footing for the model.


In [ ]:
df_model = df_clean.copy()

print("=" * 55)
print("6A: LABEL ENCODING")
print("=" * 55)

le = LabelEncoder()
for col in ['smoker', 'medical_history']:
    original = sorted(df_model[col].unique())
    df_model[col] = le.fit_transform(df_model[col])
    encoded  = sorted(df_model[col].unique())
    print(f"  '{col}': {original} → {encoded}")

print("\n sklearn's LinearRegression requires numeric inputs only.")


In [ ]:
print("=" * 55)
print(" 6B: TRAIN / TEST SPLIT  (80% / 20%)")
print("=" * 55)

X = df_model.drop(columns='annual_premium')
y = df_model['annual_premium']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f"  Training samples : {X_train.shape[0]}")
print(f"  Testing  samples : {X_test.shape[0]}")
print(f"  NaN in X_train   : {X_train.isnull().sum().sum()}  ← must be 0 before scaling")
print("\nWHY split BEFORE scaling? Scaling on the full dataset would leak")
print("test-set statistics into the training process (data leakage).")


In [ ]:
print("=" * 55)
print("6C: STANDARD SCALING   z = (x - μ) / σ")
print("=" * 55)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print(f"  sum_assured — raw mean   : {X_train['sum_assured'].mean():,.1f}")
print(f"  sum_assured — scaled mean: {X_train_scaled[:, 3].mean():.8f}  (≈ 0)")
print(f"  sum_assured — scaled std : {X_train_scaled[:, 3].std():.8f}   (≈ 1)")
print("\nAll features scaled — fair playing field for the model!")


## Step 7 — Correlation Heatmap (After Cleaning)

Let's confirm every remaining feature has meaningful correlation with the premium.

In [ ]:
corr_clean = df_model.corr()

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr_clean, dtype=bool))
sns.heatmap(corr_clean, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, vmin=-1, vmax=1, ax=ax,
            linewidths=0.5, linecolor='white',
            annot_kws={"size": 10, "weight": "bold"},
            cbar_kws={"shrink": 0.8})
ax.set_title(' Correlation Heatmap (After Cleaning)', fontsize=14, fontweight='bold', pad=15)
plt.xticks(rotation=45, ha='right'); plt.yticks(rotation=0)
plt.tight_layout(); plt.savefig('correlation_heatmap_after.png', dpi=120); plt.show()

print("\n Correlation with annual_premium (sorted):")
print(corr_clean['annual_premium'].drop('annual_premium').sort_values(ascending=False).round(3))
print("\nAll remaining features are meaningfully correlated. Dataset is model-ready!")


## Step 8 — Train the Linear Regression Model

The model learns the equation:
$$\hat{y} = \beta_0 + \beta_1 x_1 + \beta_2 x_2 + \cdots + \beta_n x_n$$

where $\hat{y}$ is the predicted **annual premium**, and each $\beta$ is learned by minimising the squared error between predicted and actual premiums (Ordinary Least Squares, handled internally by sklearn).


In [ ]:
model = LinearRegression()
model.fit(X_train_scaled, y_train)

y_pred_train = model.predict(X_train_scaled)
y_pred_test  = model.predict(X_test_scaled)

feature_names = X.columns.tolist()

print("sklearn LinearRegression trained!")
print(f"\n  Intercept (β₀) : ₹{model.intercept_:,.2f}")
print("\n  Learned Coefficients (on scaled features):")
print(f"  {'Feature':<22} {'Coefficient':>14}  {'Effect on Premium'}")
print("  " + "-"*62)
for name, coef in zip(feature_names, model.coef_):
    direction = "↑ higher premium" if coef > 0 else "↓ lower premium"
    print(f"  {name:<22} {coef:>14,.2f}  ({direction})")

print("\ne.g. 'smoker' should have a LARGE POSITIVE coefficient —")
print("matching real-world underwriting where smoking raises the premium significantly.")


## Step 9 — Manual Premium Calculation for One Applicant

Let's manually trace, using plain Python, exactly how the model arrives at a predicted premium for one fixed applicant.


In [ ]:
# ── Fixed applicant values ─────────────────────────────────────
applicant = {
    'age':               40,
    'bmi':               27.5,
    'annual_income':     800000,
    'sum_assured':       2000000,
    'policy_term_years': 20,
    'num_dependents':    2,
    'smoker':            1,      # 1 = Yes (after label encoding)
    'medical_history':   2       # 2 = None (after label encoding, alphabetical: Major=0, Minor=1, None=2)
}

print("=" * 55)
print("Applicant — Fixed Input Values")
print("=" * 55)
for k, v in applicant.items():
    print(f"  {k:<20} : {v}")

# Step 1: Build DataFrame and scale using the SAME fitted scaler
input_df     = pd.DataFrame([applicant])
input_scaled = scaler.transform(input_df)

# Step 2: Manual prediction  insurance amount = x1*age + x2*bmi + x3*income + x4*policy
intercept    = model.intercept_
coefficients = model.coef_

print("\n" + "=" * 65)
print("Manual Calculation:  ŷ = β₀ + β₁x₁ + β₂x₂ + ... + βₙxₙ")
print("Manual Calculation:  insurance amount = x1*age + x2*bmi + x3*income + x4*policy")
print("=" * 65)
print(f"\n  {'Term':<22} {'x (coef)':>12}  {'Feature':>12}  {'x * feature':>12}")
print("  " + "-"*64)

total = intercept
contributions = {}
print(f"  {'Intercept':<22} {intercept:>12.2f}  {'—':>12}  {intercept:>12.2f}")

for feat, coef, x_val in zip(feature_names, coefficients, input_scaled[0]):
    contribution = coef * x_val
    contributions[feat] = (coef, x_val, contribution)
    total += contribution
    print(f"  {feat:<22} {coef:>12.2f}  {x_val:>12.4f}  {contribution:>12.2f}")

print("  " + "-"*64)
print(f"  {'Insurance Amount':<22} {'':>27}  {total:>12.2f}")

# ── Substituted formula line ───────────────────────────────────
formula_parts = " + ".join(
    f"({coef:.2f} * {x_val:.4f})"
    for feat, (coef, x_val, _) in contributions.items()
)
print(f"\nInsurance Amount = {intercept:.2f} + {formula_parts}")
print(f"                 = ₹{total:,.2f}")

# Step 3: Verify against model.predict()
model_output = model.predict(input_scaled)[0]
print(f"\n Manual Calculation   : ₹{total:,.2f}")
print(f" model.predict()      : ₹{model_output:,.2f}")
print(f" Difference           : ₹{abs(total - model_output):.6f}  (should be ~0)")

## Step 10 — Model Evaluation

### Metrics:
| Metric | Formula | Meaning |
|---|---|---|
| **MSE** | mean of (y−ŷ)² | Average squared error in ₹²; penalises large errors heavily |
| **RMSE** | √MSE | Same unit as premium (₹) — easy to interpret |
| **R²** | 1 − SS_res/SS_tot | Fraction of premium variance explained (0 → 1) |

>**R² = 0.90** → the model explains 90% of why premiums differ between applicants.  
>**RMSE = ₹4,500** → on average, predicted premium is off by about ₹4,500.


In [ ]:
mse_train  = mean_squared_error(y_train, y_pred_train)
mse_test   = mean_squared_error(y_test,  y_pred_test)
rmse_train = np.sqrt(mse_train)
rmse_test  = np.sqrt(mse_test)
r2_train   = r2_score(y_train, y_pred_train)
r2_test    = r2_score(y_test,  y_pred_test)

print("=" * 55)
print("MODEL EVALUATION METRICS")
print("=" * 55)
print(f"  {'Metric':<15} {'Train':>15} {'Test':>15}")
print(f"  {'-'*45}")
print(f"  {'MSE':<15} {mse_train:>15,.2f} {mse_test:>15,.2f}")
print(f"  {'RMSE (₹)':<15} {rmse_train:>15,.2f} {rmse_test:>15,.2f}")
print(f"  {'R²':<15} {r2_train:>15.4f} {r2_test:>15.4f}")
print(f"\nR² = {r2_test:.2%} → model explains {r2_test:.2%} of premium variation.")
print(f"RMSE = ₹{rmse_test:,.0f} → typical prediction error in rupees.")


## Step 11 — Model Evaluation Visualizations

In [ ]:
fig = plt.figure(figsize=(18, 14))
fig.suptitle('Linear Regression — Life Insurance Premium Model Performance',
             fontsize=16, fontweight='bold', y=1.01)
gs = gridspec.GridSpec(2, 2, figure=fig, hspace=0.42, wspace=0.35)

# Plot 1: Actual vs Predicted
ax1 = fig.add_subplot(gs[0, 0])
ax1.scatter(y_test, y_pred_test, alpha=0.5, color='#3498DB',
            edgecolors='white', linewidth=0.3, s=40)
lims = [min(float(y_test.min()), y_pred_test.min()) - 2000,
        max(float(y_test.max()), y_pred_test.max()) + 2000]
ax1.plot(lims, lims, 'r--', linewidth=2, label='Perfect prediction (y = ŷ)')
ax1.set_xlabel('Actual Premium (₹)'); ax1.set_ylabel('Predicted Premium (₹)')
ax1.set_title('Actual vs Predicted Premium', fontweight='bold')
ax1.legend()
ax1.text(0.05, 0.92, f'R² = {r2_test:.3f}', transform=ax1.transAxes,
         fontsize=11, color='red', fontweight='bold',
         bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8))

# Plot 2: Residuals vs Predicted
ax2 = fig.add_subplot(gs[0, 1])
residuals = y_test.values - y_pred_test
ax2.scatter(y_pred_test, residuals, alpha=0.5, color='#E67E22',
            edgecolors='white', linewidth=0.3, s=40)
ax2.axhline(0, color='red', linewidth=2, linestyle='--', label='Zero error line')
ax2.set_xlabel('Predicted Premium (₹)'); ax2.set_ylabel('Residual (Actual − Predicted)')
ax2.set_title('Residual Plot\n(Random scatter around 0 = good model)', fontweight='bold')
ax2.legend()

# Plot 3: Residual Distribution
ax3 = fig.add_subplot(gs[1, 0])
ax3.hist(residuals, bins=35, color='#9B59B6', edgecolor='white', linewidth=0.5, alpha=0.85)
ax3.axvline(0, color='red', linewidth=2, linestyle='--')
ax3.set_xlabel('Residual (₹)'); ax3.set_ylabel('Frequency')
ax3.set_title('Residual Distribution\n(Bell curve at 0 = well-behaved errors)', fontweight='bold')

# Plot 4: Feature Coefficients
ax4 = fig.add_subplot(gs[1, 1])
coef_df = pd.DataFrame({'Feature': feature_names, 'Coefficient': model.coef_}).sort_values('Coefficient')
colors_bar = ['#E74C3C' if c < 0 else '#2ECC71' for c in coef_df['Coefficient']]
ax4.barh(coef_df['Feature'], coef_df['Coefficient'], color=colors_bar, edgecolor='white', linewidth=0.8)
ax4.axvline(0, color='black', linewidth=1)
ax4.set_title('Feature Coefficients\n(Red = lowers premium, Green = raises premium)', fontweight='bold')
ax4.set_xlabel('Coefficient Value (on scaled features)')

plt.savefig('model_evaluation_plots.png', dpi=120, bbox_inches='tight')
plt.show()
print("   Plot 1 — points near red line → accurate predictions")
print("   Plot 2 — random scatter around 0 → no systematic bias")
print("   Plot 3 — bell curve at 0 → errors are normally distributed")
print("   Plot 4 — longest green bar = strongest premium-raising factor (likely 'smoker')")


In [ ]:
# ── Best-Fit Line: age vs annual_premium ─────────────
x_bf = df_clean['age'].values.reshape(-1, 1)
y_bf = df_clean['annual_premium'].values

simple_model = LinearRegression().fit(x_bf, y_bf)
x_line = np.linspace(x_bf.min(), x_bf.max(), 300).reshape(-1, 1)
y_line = simple_model.predict(x_line)
r2_simple = r2_score(y_bf, simple_model.predict(x_bf))

fig, ax = plt.subplots(figsize=(11, 6))
ax.scatter(x_bf, y_bf, alpha=0.35, color='#3498DB', s=25,
           edgecolors='white', linewidth=0.2, label='Applicants')
ax.plot(x_line, y_line, color='#E74C3C', linewidth=2.5, label='Best-fit line (OLS)')

slope_val     = simple_model.coef_[0]
intercept_val = simple_model.intercept_
ax.text(0.05, 0.88, f'ŷ = {slope_val:.5f} × age + {intercept_val:,.0f}',
        transform=ax.transAxes, fontsize=11, color='#E74C3C', fontweight='bold',
        bbox=dict(boxstyle='round,pad=0.4', facecolor='white', alpha=0.85))
ax.text(0.05, 0.80, f'R² = {r2_simple:.3f}',
        transform=ax.transAxes, fontsize=11, color='darkgreen', fontweight='bold',
        bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.85))

ax.set_xlabel('Age (years)', fontsize=12)
ax.set_ylabel('Annual Premium (₹)', fontsize=12)
ax.set_title('Best-Fit Line — Age vs Annual Premium', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
plt.tight_layout(); plt.savefig('best_fit_line.png', dpi=120); plt.show()

print(f"   Slope = {slope_val:.5f} → every ₹1 increase in age adds ~₹{slope_val:.3f} to premium")
print(f"   R² = {r2_simple:.3f} using age ALONE.")
print(f"   Full model R² = {r2_test:.3f} — using ALL features explains FAR more of the variation!")
print(f"\n  WHY is age alone so weak (R² = {r2_simple:.3f})?")
print(f"   Because premium also depends heavily on age and medical history —")
print(f"   two applicants with the SAME age can have very different premiums")
print(f"   if one is older and the other is not. This is exactly why we need MULTIPLE")
print(f"   features (multiple linear regression) instead of relying on one variable.")


## Step 12 — Premium Prediction Function for New Applicants

In [ ]:
def predict_life_insurance_premium(age, bmi, annual_income, sum_assured,
                                   policy_term_years, num_dependents,
                                   smoker, medical_history):
    """
    Predict annual life insurance premium for a new applicant.

    Parameters:
    -----------
    age                : int    — Applicant age (18–65)
    bmi                : float  — Body Mass Index
    annual_income      : float  — Yearly income in ₹
    sum_assured        : int    — Policy payout amount in ₹ (e.g. 500000, 1000000, 2000000)
    policy_term_years  : int    — Policy duration in years (10, 15, 20, 25, 30)
    num_dependents     : int    — Number of dependents (0–5)
    smoker             : str    — 'Yes' or 'No'
    medical_history    : str    — 'None', 'Minor', or 'Major'

    Returns: predicted annual premium in ₹
    """

    smoker_map  = {'No': 0, 'Yes': 1}
    medical_map = {'Major': 0, 'Minor': 1, 'None': 2}

    if smoker not in smoker_map:
        raise ValueError("smoker must be 'Yes' or 'No'")
    if medical_history not in medical_map:
        raise ValueError(f"medical_history must be one of {list(medical_map.keys())}")

    input_df = pd.DataFrame([{
        'age':               age,
        'bmi':               bmi,
        'annual_income':     annual_income,
        'sum_assured':       sum_assured,
        'policy_term_years': policy_term_years,
        'num_dependents':    num_dependents,
        'smoker':            smoker_map[smoker],
        'medical_history':   medical_map[medical_history]
    }])

    input_scaled     = scaler.transform(input_df)
    predicted_premium = model.predict(input_scaled)[0]

    print("=" * 50)
    print(" Life Insurance Premium Prediction")
    print("=" * 50)
    print(f"  Age                : {age}")
    print(f"  BMI                : {bmi}")
    print(f"  Annual Income      : ₹{annual_income:,.0f}")
    print(f"  Sum Assured        : ₹{sum_assured:,.0f}")
    print(f"  Policy Term        : {policy_term_years} years")
    print(f"  Dependents         : {num_dependents}")
    print(f"  Smoker             : {smoker}")
    print(f"  Medical History    : {medical_history}")
    print("-" * 50)
    print(f"  Predicted Annual Premium : ₹{predicted_premium:,.2f}")
    print("=" * 50)

    return predicted_premium


# ── Example Usage ──────────────────────────────────────────────
predict_life_insurance_premium(
    age               = 40,
    bmi               = 27.5,
    annual_income     = 800000,
    sum_assured       = 2000000,
    policy_term_years = 20,
    num_dependents    = 2,
    smoker            = 'Yes',
    medical_history   = 'None'
)


In [ ]:
# Compare: same applicant but Non-Smoker
predict_life_insurance_premium(
    age               = 40,
    bmi               = 27.5,
    annual_income     = 800000,
    sum_assured       = 2000000,
    policy_term_years = 20,
    num_dependents    = 2,
    smoker            = 'No',
    medical_history   = 'None'
)
print("\n Notice how much LOWER the premium is just by changing smoker status —")
print(" this matches real-world insurance pricing, where smoking is one of the")
print(" biggest controllable risk factors.")


## Step 13 — Summary & Key Takeaways

### Problem Recap
We built a regression model to predict **annual life insurance premium** based on an applicant's age, health (BMI, medical history), lifestyle (smoking), and policy details (sum assured, term, dependents) — mirroring how real insurers price risk.

### End-to-End Pipeline:

| Step | Action | Key Insight |
|---|---|---|
| 1 | Problem statement | Premium pricing based on real-world risk factors |
| 2 | Dataset creation | 600 applicants, 9 real-world features, actuarial formula |
| 3 | EDA | Distributions, missing values, smoker vs premium insight |
| 4 | Correlation heatmap (before) | Spot zero-correlation features (`applicant_id`, `lucky_number`) |
| 5 | Data cleaning | IQR outlier removal, median/mode imputation, drop irrelevant cols |
| 6 | Feature engineering | Label encoding, train/test split, scaling |
| 7 | Correlation heatmap (after) | Confirm all features now matter |
| 8 | Model training | sklearn LinearRegression learns β coefficients |
| 9 | Manual calculation | One applicant traced step-by-step using plain Python |
| 10 | Evaluation metrics | MSE, RMSE, R² via sklearn |
| 11 | Visualisations | Actual vs predicted, residuals, best-fit line, coefficients |
| 12 | Prediction function | Ready-to-use function for new applicants |

### Real-World Takeaway
The model confirms what actuaries already know intuitively:
- **Smoking** and **major medical history** are the strongest premium-raising factors
- **Sum assured** scales the premium almost linearly
- **Age** and **policy term** add smaller, steady contributions

> This is exactly how a data-driven underwriting engine could assist (not replace) a human underwriter — fast, consistent, and explainable premium estimates.


In [ ]:
print("=" * 55)
print("     FINAL MODEL PERFORMANCE SUMMARY")
print("=" * 55)
print(f"  Dataset size after cleaning : {len(df_clean)} applicants")
print(f"  Features used               : {len(feature_names)}")
print(f"  Train / Test split          : 80% / 20%")
print()
print(f"  ┌─────────────────────────────────────────┐")
print(f"  │  Training R²  :  {r2_train:.4f}                 │")
print(f"  │  Test     R²  :  {r2_test:.4f}                 │")
print(f"  │  Test  RMSE   :  ₹{rmse_test:>10,.0f}            │")
print(f"  └─────────────────────────────────────────┘")
print()
print("   You now understand how a Linear Regression model can power")
print("   real-world insurance premium pricing — from raw applicant data")
print("   to a fully explainable, data-driven premium estimate!")
